# N4 bias correction — ANTsImage interface

Use `antstorch.n4_bias_field_correction` with an ANTsImage and an ANTsImage mask.
This is ANTsTorch's image wrapper around its tensor N4 engine, not a call to
`ants.n4_bias_field_correction`. Outputs retain the input geometry automatically.
The real r16 tutorial image downloads on first use; no pretrained weights are needed.

Compare [the tensor interface](n4_bias_correction_tensor.ipynb), which exposes
conversion, device placement and `ImageDomain` explicitly. Both examples use
the same input resolution and N4 parameters.

Install ANTsTorch and notebook tools from the repository root before opening this notebook:
```bash
python -m pip install -e '.[notebooks]'
python -m ipykernel install --user --name python3 --display-name "Python 3 (ANTsTorch)"
```
Select that kernel and run the cells in order. All examples default to CPU.

In [ ]:
from time import perf_counter
_example_started = perf_counter()

import torch  # Import before ANTs to avoid OpenMP conflicts on macOS.
import numpy as np
import ants
import antstorch

# Display figures inside Jupyter/VS Code even if MPLBACKEND was set to Agg.
from IPython import get_ipython
shell = get_ipython()
if shell is not None:
    shell.run_line_magic("matplotlib", "inline")
import matplotlib.pyplot as plt

device = "cpu"
antstorch.set_default_device(device)

# Synchronize the selected accelerator so timings include completed GPU work.
from contextlib import contextmanager
_example_operation_times = []

def _synchronize_example_device():
    selected = torch.device(device)
    if selected.type == "cuda":
        torch.cuda.synchronize(selected)
    elif selected.type == "mps":
        torch.mps.synchronize()

@contextmanager
def time_step(label):
    _synchronize_example_device()
    started = perf_counter()
    try:
        yield
    finally:
        _synchronize_example_device()
        seconds = perf_counter() - started
        _example_operation_times.append((label, seconds))
        print(f"{label}: {seconds:.4f} s")

In [ ]:
reference = ants.image_read(ants.get_ants_data("r16")).clone("float")
reference = ants.resample_image(reference, (128, 128), use_voxels=True)
mask = ants.get_mask(reference)
options = dict(shrink_factor=2, convergence={"iters": [10, 10], "tol": 1e-7},
               spline_param=(4, 4), number_of_histogram_bins=64)
with time_step("N4 correction — ANTsImage"):
    corrected = antstorch.n4_bias_field_correction(reference, mask=mask, device=device, **options)
with time_step("N4 bias field — ANTsImage"):
    bias = antstorch.n4_bias_field_correction(
        reference, mask=mask, return_bias_field=True, device=device, **options,
    )
assert ants.is_image(corrected) and ants.is_image(bias)
for image in (corrected, bias):
    assert image.shape == reference.shape and np.isfinite(image.numpy()).all()
    assert ants.image_physical_space_consistency(image, reference)
assert (bias.numpy() > 0).all()

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3), constrained_layout=True)
for ax, image, title in zip(axes, [reference, corrected, bias],
                            ["Tutorial r16", "N4 corrected", "Estimated bias"]):
    view = ax.imshow(image.numpy().T, origin="lower", cmap="gray")
    ax.set_title(title)
    fig.colorbar(view, ax=ax, shrink=0.7)
plt.show()

The wrapper handles array-axis reversal and restores spacing, origin and direction.
Real r16 anatomy has no known bias-free reference, so this example checks output
structure and displays the estimated field rather than asserting a reduction in
global intensity variation. The short schedule is illustrative.

## Execution time

Run **Restart Kernel and Run All Cells** for a comparable end-to-end duration.
The total starts at the first setup cell and includes imports, downloads, data
preparation, computation, plotting and file writes. Kernel startup is excluded;
manual pauses between cells are included. Operation timings include any
preprocessing or asset downloads performed inside the called function. First
runs can be slower because caches are empty. Timings describe this run on the
selected device, not a fixed performance estimate for other machines.

In [ ]:
_synchronize_example_device()
total_seconds = perf_counter() - _example_started
operation_seconds = sum(seconds for _, seconds in _example_operation_times)
print(f"Total elapsed time: {total_seconds:.2f} s ({total_seconds / 60:.2f} min)")
print(f"Timed operations: {operation_seconds:.4f} s on {device}")